# Trade bot experiment, base in AI signals

### Import libraries

In [1]:
import numpy as np
import pandas as pd
import os
import  datetime
import json
import time



In [2]:
import os, sys
processing_source_path = os.path.abspath('./../AI/Processing/')
if(processing_source_path not in sys.path):
    sys.path.append(processing_source_path)
from DataLoaderPipeline import scrapingHistoricalData, FeaturesDataGenerator

from MarketIndicators import ComputSignalGains

import  ProcessingPipeline as pp

In [3]:
CSG=ComputSignalGains()

### get recomendations

In [4]:
def get_recommendation_history(model_name: str, crypto: str):
    file_path = f'./../AI/Classification/Real_Time_Inference/Recommendations/{model_name}_{crypto}_recommendation.csv'
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        history = []
        for index, row in df.iterrows():
            history.append({
                'Date': row['Date'],
                'Time': row['Time'],
                'recommendation': row['recommendation'],
                'percentage': row['percentage'],
                'Price': row['Price'],
            })
        return history
    else:
        return None

In [5]:
symbol="SOL"
list_of_models = ['CNN_MultiHead_2D']
used_model= list_of_models[0]
last_timestamp = pd.Timestamp('2020-02-05 16:00:00')

classifcation_model_path = "./../AI/Classification/Experiments/Cryptos/models/"
sufix = 'last'
cryptos = ['BTC', 'ETH', 'SOL','XRP','ADA']

In [6]:
for symbol in cryptos:

    if 'CNN' in used_model:
        model_type='CNN'

    AI_recomendations = get_recommendation_history(model_name=model_type, crypto=symbol)
    # get the last model recommendation 
    last_recomendation = AI_recomendations[-1]
    print(last_recomendation)
    #get the recommendation timestamp
    last_recomendation_timestamp = pd.to_datetime(f"{last_recomendation['Date']} {last_recomendation['Time']}")

    # get the model lookback
    checkpoint_filepath =f'{classifcation_model_path}model_{list_of_models[0]}_crypto_{symbol}_best'
    # JSON file
    if os.path.exists(checkpoint_filepath) == False:
        checkpoint_filepath =f'{classifcation_model_path}/model_CNN_restnet_MultiHead_2D_crypto_General_{sufix}'

    with open(f'{checkpoint_filepath}/config.json', 'r') as file:
        parameters = json.load(file)

    lookback = parameters['lookback']

    #get the recommendation timestamp
    last_recomendation_timestamp = pd.to_datetime(f"{last_recomendation['Date']} {last_recomendation['Time']}")

    # scraping the last data 
    SHD=scrapingHistoricalData()
    date_now = datetime.datetime.now()
    date_innit = date_now- datetime.timedelta(days=100)
    cryptos_df = SHD.get_crypto_historical_data([symbol], '4h', date_innit, date_now.strftime('%Y-%m-%d'))

    # get idx that is equal to the AI recomendation timestamp
    last_idx=cryptos_df[cryptos_df['Date']==last_recomendation_timestamp].index.values[0]
    # get the last N values of loockback
    last_recomendation_window=cryptos_df.iloc[last_idx-lookback:last_idx+1]

    high = last_recomendation_window['Close'].max()
    low = last_recomendation_window['Close'].min()
    middle = last_recomendation_window['Close'].median()
    input_price = last_recomendation_window['Close'].iloc[-1]
    
    if last_recomendation['recommendation']!="Hold":

        if last_recomendation['recommendation']!="Sell":
            resultados_fibonacci = CSG.calculate_Gains_Fibonacci(input_price, low, high, side=last_recomendation['recommendation'])
        else:
            resultados_fibonacci = CSG.calculate_Gains_Fibonacci(input_price, low, high, side=last_recomendation['recommendation'])

        print(f"Suggested trade of {symbol}: {last_recomendation['recommendation']} | Price: {input_price}>> Timestamp: {last_recomendation_timestamp}")
        print(f"**AI Trust percentage:** {last_recomendation['percentage']}% 🔍")

        print("-----------------------------------------------------------------------------")
        print("**Scenario aggressive:** 🚀")
        print("-------------------------------")
        print(f"⚠️ **Stop Loss:** {resultados_fibonacci['aggressive']['stop_loss']}")
        print(f"🎯 **Alvo:** {resultados_fibonacci['aggressive']['target']}")
        print("-------------------------------")

        print("\n**Scenario conservative:** 🚨")
        print("-------------------------------")
        print(f"⚠️ **Stop Loss:** {resultados_fibonacci['conservative']['stop_loss']}")
        print(f"🎯 **Alvo:** {resultados_fibonacci['conservative']['target']}")
        print("-------------------------------")

        print("\n**Scenario moderate:** 📊")
        print("-------------------------------")
        print(f"⚠️ **Stop Loss:** {resultados_fibonacci['moderate']['stop_loss']}")
        print(f"🎯 **Alvo:** {resultados_fibonacci['moderate']['target']}")
        print("-------------------------------")
        
    # Update the last timestamp
    last_timestamp = last_recomendation_timestamp

{'Date': '2025-06-11', 'Time': '16:00:00', 'recommendation': 'Hold', 'percentage': 0.9831819, 'Price': 108546.49}
{'Date': '2025-06-11', 'Time': '16:00:00', 'recommendation': 'Hold', 'percentage': 0.7052168, 'Price': 2789.72}
{'Date': '2025-06-11', 'Time': '16:00:00', 'recommendation': 'Hold', 'percentage': 0.9805697, 'Price': 162.02}
{'Date': '2025-06-11', 'Time': '16:00:00', 'recommendation': 'Hold', 'percentage': 0.94745356, 'Price': 2.2817}
{'Date': '2025-06-11', 'Time': '16:00:00', 'recommendation': 'Buy', 'percentage': 0.54276866, 'Price': 0.6999}
Suggested trade of ADA: Buy | Price: 0.7042>> Timestamp: 2025-06-11 16:00:00
**AI Trust percentage:** 0.54276866% 🔍
-----------------------------------------------------------------------------
**Scenario aggressive:** 🚀
-------------------------------
⚠️ **Stop Loss:** 0.6605
🎯 **Alvo:** 0.8043
-------------------------------

**Scenario conservative:** 🚨
-------------------------------
⚠️ **Stop Loss:** 0.6901
🎯 **Alvo:** 0.7543
-----

In [7]:
resultados_fibonacci

{'conservative': {'target': 0.7543, 'stop_loss': 0.6901},
 'moderate': {'target': 0.7661, 'stop_loss': 0.6723},
 'aggressive': {'target': 0.8043, 'stop_loss': 0.6605}}